# Stratification Timing: runtime vs. dimension $d$

This notebook visualizes a benchmark of tensor **stratification** applied to the SphereLab
hidden-sphere tensor: a cubical $d \times d \times d$ tensor is built to hide a sphere, and we
measure how long it takes to recover the stratification as $d$ grows ($d = 10, 15, 20, \dots$).

Each run is split into two timed phases:

- **`der_seconds`** — solving the *derivation* (the null-space problem) for the tensor,
- **`strat_seconds`** — computing the *stratification* itself given that derivation
  (eigendecomposition per axis plus contractions).

The benchmark crosses two precision settings (`Float32` solved to `tol=1e-8`, `Float64` solved
to `tol=1e-16`) with two operator spaces (`universal`, the general unrestricted chisel, and
`symmetric`, the chisel restricted to symmetric operators). Several solver methods are tried in
each operator space, and not every method exists in both spaces — that asymmetry is expected,
not missing data.

**60 s drop-out rule:** a method is dropped from all larger $d$ once its runtime exceeds a 60 s
budget. A curve that simply stops partway across the $x$-axis is the headline result for that
method, not a data gap.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.width", 120)

# One color per method, reused across every figure in this notebook.
METHOD_ORDER = [
    "SylverLining/Auto",
    "SylverLining/SVD",
    "Auto",
    "QuickDer",
    "QuickDer3",
    "SymmetricGram",
]
COLOR_MAP = {
    "SylverLining/Auto": "#4C78A8",
    "SylverLining/SVD": "#F58518",
    "Auto": "#54A24B",
    "QuickDer": "#E45756",
    "QuickDer3": "#B279A2",
    "SymmetricGram": "#72B7B2",
}

df = pd.read_csv("stratify-timing.csv")
df.head()

In [ ]:
n_total = len(df)
ok = df[df.status == "ok"].copy()
n_error = n_total - len(ok)
print(f"{n_error} of {n_total} rows errored out; {len(ok)} rows remain with status == 'ok'.")
if n_error:
    display(df.loc[df.status != "ok", ["d", "eltype", "ops", "method", "status"]])

In [ ]:
# Pivot total_seconds by d (rows) x method (columns), one table per (eltype, ops).
for eltype in ["Float32", "Float64"]:
    for ops in ["universal", "symmetric"]:
        sub = ok[(ok.eltype == eltype) & (ok.ops == ops)]
        if sub.empty:
            continue
        pivot = sub.pivot_table(index="d", columns="method", values="total_seconds")
        cols = [m for m in METHOD_ORDER if m in pivot.columns]
        pivot = pivot[cols]
        print(f"total_seconds -- eltype={eltype}, ops={ops}")
        display(pivot)

In [ ]:
fig_total = px.line(
    ok.sort_values("d"),
    x="d",
    y="total_seconds",
    color="method",
    facet_row="eltype",
    facet_col="ops",
    category_orders={"method": METHOD_ORDER},
    color_discrete_map=COLOR_MAP,
    markers=True,
    log_y=True,
    template="plotly_white",
    title="Total stratification runtime vs. dimension d",
    labels={"d": "axis dimension d", "total_seconds": "seconds"},
)
fig_total.update_yaxes(matches=None, showticklabels=True)
fig_total.show()

In [ ]:
# Split each run into its derivation-solve and stratification phases so the two can be
# compared directly: one line per (method, phase), same color per method, dashed vs. solid
# for the phase.
split = ok.melt(
    id_vars=["d", "eltype", "ops", "method"],
    value_vars=["der_seconds", "strat_seconds"],
    var_name="phase",
    value_name="seconds",
).sort_values("d")

fig_split = px.line(
    split,
    x="d",
    y="seconds",
    color="method",
    line_dash="phase",
    facet_row="eltype",
    facet_col="ops",
    category_orders={"method": METHOD_ORDER, "phase": ["der_seconds", "strat_seconds"]},
    color_discrete_map=COLOR_MAP,
    markers=True,
    log_y=True,
    template="plotly_white",
    title="Derivation solve (solid) vs. stratification (dashed) time vs. dimension d",
    labels={"d": "axis dimension d", "seconds": "seconds"},
)
fig_split.update_yaxes(matches=None, showticklabels=True)
fig_split.show()

In [ ]:
ok["der_frac"] = ok.der_seconds / ok.total_seconds

fig_frac = px.line(
    ok.sort_values("d"),
    x="d",
    y="der_frac",
    color="method",
    facet_col="ops",
    category_orders={"method": METHOD_ORDER},
    color_discrete_map=COLOR_MAP,
    markers=True,
    template="plotly_white",
    title="Fraction of total time spent solving the derivation vs. dimension d",
    labels={"d": "axis dimension d", "der_frac": "der_seconds / total_seconds"},
)
fig_frac.update_yaxes(range=[0, 1.05])
fig_frac.show()

In [ ]:
fig_acc = px.line(
    ok.sort_values("d"),
    x="d",
    y="lsq_err",
    color="method",
    facet_row="eltype",
    facet_col="ops",
    category_orders={"method": METHOD_ORDER},
    color_discrete_map=COLOR_MAP,
    markers=True,
    log_y=True,
    template="plotly_white",
    title="Recovered-sphere reconstruction error vs. dimension d",
    labels={"d": "axis dimension d", "lsq_err": "lsq_err (relative reconstruction error)"},
)
fig_acc.update_yaxes(matches=None, showticklabels=True)
fig_acc.show()

**Reading the accuracy plot:** in the `universal` operator space the chisel is unrestricted,
so it finds a derivation space with `nullity` well above 3 on this input. A random combination
drawn from that larger space does not stratify to the sphere, so `lsq_err` sits near 1 in the
`universal` facets — that is expected, not a bug. The `symmetric` operator space is the
well-posed one for this problem: it should reach `lsq_err` of roughly `1e-6` in `Float32` and
roughly `1e-14` in `Float64`.

## Reproducing this benchmark

From the repository root, using the project's Julia wrapper (never bare `julia`):

```
bench/jl timing/StratifyTiming.jl 200 60
```

This regenerates `timing/stratify-timing.csv` (max dimension 200, 60 s per-method time budget),
which this notebook reads directly.